# Notebook 2: Accessing Materials Data — Materials Project

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Query the **Materials Project** database using the `mp-api` Python client
- Search for materials by formula, elements, and properties
- Download and inspect crystal structures
- Understand SCIGEN's training data and why we need generative models

> **Prerequisites:** Run `00_setup.ipynb` first. An MP API key is optional — we provide fallback data.

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg])

_ensure('pymatgen')
_ensure('mp_api', 'mp-api')

# Clone the repo if not present (needed for dataset files)
import os, shutil
REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 2.1 The Materials Project

The [Materials Project](https://materialsproject.org) is the largest open database of
computed materials properties, with ~150,000 inorganic crystal structures.

Each entry includes:
- Crystal structure (atomic positions, lattice)
- Formation energy, band gap, energy above hull
- Magnetic properties, elastic properties, and more

SCIGEN was trained on the **MP-20** subset: structures with ≤20 atoms per unit cell.

---
## 2.2 Setting up the MP API client

To query the Materials Project live, you need a free API key from
[materialsproject.org/api](https://materialsproject.org/api).

If you don't have one, skip to **Section 2.4** where we use pre-downloaded data.

In [ ]:
# Set your API key here (or leave as None to use fallback data)
MP_API_KEY = None  # e.g., 'your_api_key_here'

USE_MP_API = MP_API_KEY is not None

if USE_MP_API:
    from mp_api.client import MPRester
    mpr = MPRester(MP_API_KEY)
    print('MP API client ready.')
else:
    print('No API key provided — using pre-downloaded data.')
    print('(To enable live queries, get a free key at materialsproject.org/api)')

---
## 2.3 Querying the Materials Project

Let's search for manganese-containing materials — the kind of materials
we'll later generate with SCIGEN.

In [ ]:
if USE_MP_API:
    # Search for stable Mn-containing materials
    results = mpr.materials.summary.search(
        elements=['Mn'],
        energy_above_hull=(0, 0.05),  # near-stable materials
        num_sites=(1, 20),  # small unit cells (like SCIGEN's training data)
        fields=['material_id', 'formula_pretty', 'formation_energy_per_atom',
                'energy_above_hull', 'nsites', 'symmetry']
    )
    print(f'Found {len(results)} near-stable Mn-containing materials with ≤20 atoms/cell')
    print()
    for r in results[:10]:
        print(f'  {r.material_id}  {r.formula_pretty:<15s}  '
              f'Ef={r.formation_energy_per_atom:.3f} eV/atom  '
              f'Ehull={r.energy_above_hull:.3f} eV  '
              f'nsites={r.nsites}')
else:
    print('(Skipping live query — no API key)')

In [ ]:
if USE_MP_API:
    # Download a specific structure
    struct_mp = mpr.get_structure_by_material_id('mp-35')  # Mn metal
    print(f'Downloaded: Mn metal (mp-35)')
    print(struct_mp)
else:
    print('(Skipping live download — no API key)')

---
## 2.4 Exploring SCIGEN's training data

Even without an API key, we can explore the MP-20 dataset that SCIGEN was trained on.
This data is included in the repository.

In [ ]:
import pandas as pd
import os

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')

train_df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'train.csv'))
val_df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'val.csv'))
test_df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'test.csv'))

print(f'MP-20 dataset:')
print(f'  Train: {len(train_df):,} structures')
print(f'  Val:   {len(val_df):,} structures')
print(f'  Test:  {len(test_df):,} structures')
print(f'  Total: {len(train_df) + len(val_df) + len(test_df):,} structures')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Formation energy distribution
axes[0].hist(train_df['formation_energy_per_atom'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Formation energy (eV/atom)')
axes[0].set_ylabel('Count')
axes[0].set_title('Formation energy distribution')

# Band gap distribution
axes[1].hist(train_df['band_gap'], bins=60, color='coral', edgecolor='white')
axes[1].set_xlabel('Band gap (eV)')
axes[1].set_ylabel('Count')
axes[1].set_title('Band gap distribution')

# Energy above hull
axes[2].hist(train_df['e_above_hull'], bins=60, color='seagreen', edgecolor='white',
             range=(0, 0.5))
axes[2].set_xlabel('Energy above hull (eV/atom)')
axes[2].set_ylabel('Count')
axes[2].set_title('Stability (e_above_hull)')

plt.tight_layout()
plt.show()

In [ ]:
# What elements are most common in the training data?
from collections import Counter
import ast

all_elements = []
for elem_str in train_df['elements']:
    try:
        elems = ast.literal_eval(elem_str)
        all_elements.extend(elems)
    except:
        pass

elem_counts = Counter(all_elements).most_common(20)

fig, ax = plt.subplots(figsize=(10, 4))
elements, counts = zip(*elem_counts)
ax.bar(elements, counts, color='steelblue', edgecolor='white')
ax.set_ylabel('Number of structures')
ax.set_title('Top 20 elements in MP-20 training data')
plt.tight_layout()
plt.show()

---
## 2.5 How many kagome materials exist?

This is the motivating question for SCIGEN: if we search the known databases,
how many materials with a kagome sublattice do we find?

The answer is: **very few.** Kagome magnets are rare in nature, but they are
extremely interesting for physics (frustrated magnetism, spin liquids, flat bands).

SCIGEN's goal is to **generate new candidate materials** with these special lattice
geometries — expanding the search space far beyond what's been experimentally observed.

In [ ]:
# Count Mn-containing materials in the training data
mn_count = 0
for elem_str in train_df['elements']:
    try:
        if 'Mn' in ast.literal_eval(elem_str):
            mn_count += 1
    except:
        pass

print(f'Mn-containing structures in training data: {mn_count} / {len(train_df)}')
print(f'That\'s {mn_count/len(train_df)*100:.1f}% of the dataset.')
print()
print('Of these, even fewer have a kagome sublattice.')
print('This is exactly why we need generative models —')
print('to explore regions of materials space that are underrepresented in databases.')

---
## Key takeaways

1. **Materials databases are large** (~150K structures in MP) but they mostly cover
   well-known, thermodynamically stable materials.
2. **Exotic geometries are rare** — kagome, honeycomb, and other frustrated lattices
   appear in only a handful of known structures.
3. **Generative models** can help explore the vast space of *possible* materials
   that haven't been made yet.

In the next notebook, we'll learn how these generative models work.

---
## What's next?

Proceed to **Notebook 03: Generative AI Concepts** to learn how diffusion models generate crystal structures.